# 02 · Agregações de Negócio (Caso A)

🎯 **Objetivo:** Consolidar o uso de `groupBy`/`agg`/`orderBy` para responder perguntas de negócio com dados agregados.

**Teoria:** docs/04-dataframes-catalyst-tungsten.md

Ainda `local[*]`. Este notebook fica só em `vendas` — sem joins — para você se concentrar no padrão **agrupar → agregar → ordenar** antes de complicar com múltiplas tabelas (isso vem no notebook 03).

---
### 📊 O que são agregações?

Agregações transformam **muitas linhas** em **poucas linhas de resumo**. Exemplos do dia a dia:
- Total de vendas por mês
- Média de salário por cargo
- Contagem de funcionários por empresa

No Spark, o padrão é sempre: **groupBy → agg → orderBy**.
Vamos praticar!


In [ ]:
import sys

sys.path.insert(0, "../scripts")
from lab_utils import get_local_session, layer_path

# Cria a SparkSession local e carrega apenas a tabela de vendas
spark = get_local_session("02-agregacoes-negocio")
vendas = spark.read.parquet(layer_path("local", "bronze", "vendas"))
# show() dispara a leitura e exibe as primeiras 5 linhas
vendas.show(5)

✅ **Dados carregados.** A tabela `vendas` contém transações com valor, data (ano/mês/dia) e referências aos funcionários. Vamos explorar esses dados com agregações.


## Total de vendas por ano/mês

Como a receita evoluiu mês a mês? Esta é a pergunta clássica de **série temporal** que o `groupBy` resolve.

🧠 **Padrão:** `groupBy(colunas_agrupadoras).agg(função_agregadora)`. O Spark:
1. Embaralha os dados para reunir linhas do mesmo grupo (shuffle)
2. Aplica a função de agregação dentro de cada grupo
3. Retorna um DataFrame com uma linha por grupo

In [ ]:
from pyspark.sql.functions import col
# Importa a função sum com alias para não conflitar com sum() do Python
from pyspark.sql.functions import sum as spark_sum

# Agrupa por ano e mês, soma o valor das vendas, ordena cronologicamente
vendas_por_periodo = (
    vendas.groupBy("ano", "mes")
    .agg(spark_sum("valor").alias("total_vendas"))
    .orderBy("ano", "mes")
)
# Exibe todos os 24 meses (2 anos de dados)
vendas_por_periodo.show(24)

📌 **Interpretação da saída:**

- Cada linha representa um mês de vendas.
- A coluna `total_vendas` é a soma de todas as vendas daquele período.
- A ordenação por ano/mês revela a **sazonalidade** do negócio.

💡 **Dica:** Repare que usamos `spark_sum("valor")` e não `sum("valor")`. Isso porque o Python já tem uma função `sum()` embutida. O alias evita conflito.


## Ticket médio e volume de transações

Duas métricas fundamentais para qualquer negócio:
- **Ticket médio:** qual o valor típico de cada venda?
- **Volume:** quantas transações aconteceram no total?

💡 **Dica:** `avg()` e `count()` são funções de agregação embutidas no Spark SQL. Quando usadas sem `groupBy`, elas agregam **todas as linhas** em uma única linha de resultado.

In [ ]:
from pyspark.sql.functions import avg, count

# Agregação global (sem groupBy) — uma única linha de resultado
vendas.agg(
    avg("valor").alias("ticket_medio"),
    count("*").alias("total_transacoes"),
# count("*") conta todas as linhas, incluindo nulos
).show()

📌 **O que o ticket médio nos diz?**

- Um ticket médio baixo pode indicar vendas de volume alto (muitas vendas de pouco valor).
- Um ticket médio alto pode indicar vendas mais esporádicas mas de maior valor.
- O `count(*)` conta todas as linhas — incluindo valores nulos em qualquer coluna.

🧠 **Desafio:** Como você calcularia o ticket médio **por ano**?


## As 10 maiores vendas individuais

Ao contrário das agregações anteriores, aqui **não agrupamos** — apenas ordenamos todas as linhas pela coluna `valor` em ordem decrescente e exibimos as 10 primeiras.

🧠 **Por quê?** `orderBy().show(10)` com `desc()` é um padrão muito usado para encontrar outliers, registros suspeitos ou apenas conhecer os extremos dos dados.

In [ ]:
# Projeta as colunas relevantes e ordena por valor decrescente
maiores_vendas = (
    vendas.select("id_venda", "id_funcionario", "valor", "ano", "mes", "dia")
    .orderBy(col("valor").desc())
)
# Exibe apenas as 10 maiores — não carrega todo o dataset na memória
maiores_vendas.show(10)

📌 **Análise:** As 10 maiores vendas revelam os valores máximos do dataset. Compare com o ticket médio — a diferença mostra a **dispersão** dos dados.

⚠️ **Atenção:** `orderBy().show(10)` é eficiente porque o Spark usa uma otimização chamada **TakeOrderedAndProject** — ele encontra os 10 maiores sem precisar ordenar o dataset inteiro.


## Vendas por ano: total, contagem e ticket médio

`agg` aceita **várias agregações de uma vez** — não precisa de uma chamada por métrica. Isso é mais eficiente porque o Spark processa todas as agregações em uma única passada sobre os dados.

💡 **Dica:** Use `alias()` para dar nomes legíveis às colunas agregadas. Caso contrário, o Spark usa nomes genéricos como `sum(valor)`.

In [ ]:
# Múltiplas agregações em um único groupBy — processamento eficiente
resumo_anual = (
    vendas.groupBy("ano")
    .agg(
        spark_sum("valor").alias("total_vendas"),
        count("*").alias("numero_vendas"),
        avg("valor").alias("ticket_medio"),
    )
    .orderBy("ano")
)
resumo_anual.show()

📌 **Vantagem de múltiplas agregações:**

- Em vez de rodar 3 consultas separadas (uma para total, uma para contagem, uma para média), fazemos tudo em **um único `agg`**.
- O Spark Catalyst Optimizer consegue fundir as operações em um único estágio de shuffle.
- Resultado: uma tabela resumo por ano com todas as métricas lado a lado.

💡 **Dica:** Você pode usar `min()`, `max()`, `stddev()`, `approx_count_distinct()` e dezenas de outras funções dentro do `agg`.


In [ ]:
# Encerra a SparkSession — libera threads e memória
spark.stop()

---
🎉 **Agregações concluídas!** Você aprendeu:

- `groupBy` para agrupar dados por colunas categóricas
- `agg` com `sum`, `avg`, `count` para calcular métricas
- `orderBy` com `desc()` para ordenar resultados
- Múltiplas agregações em uma única passada

▶️ **Próximo:** Notebook 03 — Joins entre Vendas, Funcionários e Empresas
